# Outlier Detection and Robust Handling

An **outlier** is a data point that differs significantly from other observations. It is a value that lies abnormally far away from the rest of your data. 

Outliers can happen for two reasons:
1. **Data Entry Errors**: Someone accidentally typed an extra zero (e.g., age = 250). These should be removed.
2. **True Anomalies**: A legitimate but extreme event (e.g., a customer actually *did* spend $50,000 on one order). These require careful handling so they don't skew your model.

If you don't handle outliers, algorithms like Linear Regression will completely change their trajectory just to accommodate one single extreme data point.

Let's set up a sandbox with a dataset containing some hidden outliers!

In [1]:
import pandas as pd
import numpy as np

# Create a dataset of house prices
data = {
    'house_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'bedrooms': [3, 4, 3, 2, 4, 3, 3, 50], # 50 bedrooms is highly unusual!
    'price': [250000, 300000, 280000, 210000, 320000, 275000, 260000, 5000000] # $5M is an outlier here
}

df = pd.DataFrame(data)

print("--- Original House Data ---")
display(df)

--- Original House Data ---


,house_id,bedrooms,price
0,1,3,250000
1,2,4,300000
2,3,3,280000
3,4,2,210000
4,5,4,320000
5,6,3,275000
6,7,3,260000
7,8,50,5000000


# 1. Detection Method 1: The Z-Score
The **Z-Score** is a statistical measurement that tells you exactly how many *Standard Deviations* a data point is away from the mean (average). 

In a normal distribution, 99.7% of all data falls within 3 standard deviations. Therefore, a common rule of thumb in Data Science is: **If a Z-Score is greater than 3 or less than -3, it is an outlier.**

In [2]:
# Create a copy to work with
df_zscore = df.copy()

# Calculate the Z-Score for the 'price' column manually using Pandas
mean_price = df_zscore['price'].mean()
std_price = df_zscore['price'].std()

df_zscore['price_zscore'] = (df_zscore['price'] - mean_price) / std_price

print("--- Data with Z-Scores ---")
display(df_zscore[['house_id', 'price', 'price_zscore']])

# Filter out the outliers (Keep only rows where Z-Score is between -3 and 3)
# Note: Because our dataset is so small, our extreme outlier only hit a Z-score of ~2.4, 
# but in a dataset of 10,000 rows, $5M would easily exceed 3.0!

--- Data with Z-Scores ---


,house_id,price,price_zscore
0,1,250000,-0.365871
1,2,300000,-0.335973
2,3,280000,-0.347932
3,4,210000,-0.389789
4,5,320000,-0.324014
5,6,275000,-0.350922
6,7,260000,-0.359891
7,8,5000000,2.474392


*(The Danger of Z-Scores: The Z-Score relies on the Mean. But remember, the Mean itself is heavily pulled by outliers! This creates a chicken-and-egg problem where massive outliers can hide themselves by inflating the mean. Because of this, we often prefer the next method.)*

# 2. Detection Method 2: The IQR (Interquartile Range)
The **IQR method** is incredibly robust because it completely ignores the mean. Instead, it relies on percentiles (which are not affected by extreme values).

* **Q1 (25th Percentile)**: The value where 25% of the data is smaller.
* **Q3 (75th Percentile)**: The value where 75% of the data is smaller.
* **IQR**: The distance between Q1 and Q3 (`Q3 - Q1`).

**The Rule**: Anything lower than `Q1 - (1.5 * IQR)` or higher than `Q3 + (1.5 * IQR)` is an outlier.

In [3]:
# Create a copy to work with
df_iqr = df.copy()

# 1. Calculate Q1 and Q3
Q1 = df_iqr['price'].quantile(0.25)
Q3 = df_iqr['price'].quantile(0.75)

# 2. Calculate the IQR
IQR = Q3 - Q1

# 3. Define the Lower and Upper Bounds
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

print(f"Normal Price Range: ${lower_bound:,.2f} to ${upper_bound:,.2f}")

# 4. Filter the data to find the outliers
outliers = df_iqr[(df_iqr['price'] < lower_bound) | (df_iqr['price'] > upper_bound)]

print("\n--- Identified Outliers using IQR ---")
display(outliers)

Normal Price Range: $186,250.00 to $376,250.00

--- Identified Outliers using IQR ---


,house_id,bedrooms,price
7,8,50,5000000


# 3. Handling Outliers: Trimming (Deletion)
Once we have identified the outliers (using Z-Score or IQR), the easiest way to handle them is to just drop them from the dataset entirely.

In [4]:
# Keep only the rows that fall INSIDE our calculated IQR bounds
df_trimmed = df_iqr[(df_iqr['price'] >= lower_bound) & (df_iqr['price'] <= upper_bound)]

print("--- Data after Trimming (Dropping Outliers) ---")
display(df_trimmed)

--- Data after Trimming (Dropping Outliers) ---


,house_id,bedrooms,price
0,1,3,250000
1,2,4,300000
2,3,3,280000
3,4,2,210000
4,5,4,320000
5,6,3,275000
6,7,3,260000


*(Just like dropping missing values, trimming is great, but you throw away data. If the outlier was a highly valuable $5M customer, you just deleted them from your model!)*

# 4. Handling Outliers: Capping (Winsorization)
If you don't want to throw the data away, you can "Cap" it. This process is known in statistics as **Winsorization**.

Instead of deleting the $5M house, we simply say: *"Any house priced above our upper boundary will just be recorded as the upper boundary."* It reigns in the extreme values without losing the row. 

We can do this easily using Pandas' `.clip()` method!

In [5]:
# Create a copy for capping
df_capped = df.copy()

# Cap the 'price' column using our IQR bounds from earlier
df_capped['price'] = df_capped['price'].clip(lower=lower_bound, upper=upper_bound)

print(f"Capping all prices at a maximum of ${upper_bound:,.2f}...")
print("\n--- Data after Capping (Winsorization) ---")
display(df_capped)

Capping all prices at a maximum of $376,250.00...

--- Data after Capping (Winsorization) ---


,house_id,bedrooms,price
0,1,3,250000
1,2,4,300000
2,3,3,280000
3,4,2,210000
4,5,4,320000
5,6,3,275000
6,7,3,260000
7,8,50,376250


*(Notice House #8 wasn't deleted! It kept its 50 bedrooms, but its price was gently pulled down from $5,000,000 to the maximum acceptable boundary of $357,500. It is still the most expensive house in the dataset, but it is no longer mathematically destructive!)*

---

## Real-World Use Case or Analogy:
Think of Outliers using the famous **Billionaire in a Bar Analogy**:

* **The Setup**: There are 10 average people sitting in a bar. They each make $50,000 a year. The **Mean** (average) income of the bar is $50,000.
* **The Outlier Appears**: Bill Gates walks into the bar. He makes $1 Billion a year.
* **The Problem (Why Z-Scores can fail)**: If you calculate the new Mean, the "average" person in this bar now makes $90,954,545 a year! If a Machine Learning model looks at this, it will assume everyone in the bar is flying private jets. The one extreme outlier completely destroyed the model's understanding of reality.
* **The Solution (IQR)**: The IQR method uses the Median (the middle person). Even with Bill Gates in the room, the middle person's income is still exactly $50,000. The IQR flags Bill Gates as a mathematical anomaly, allowing the Data Scientist to either ask him to leave (Trimming) or temporarily record his income as $100,000 (Capping) so the math works again.

---